In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

# ========= 1) Excel 读列 =========
file_path = r"your_data.xlsx"
sheet_name = "Classify"
feature_cols = ["x1", "x2", "x3", "x4"]
target_col = "label"

df = pd.read_excel(file_path, sheet_name=sheet_name)
X = df[feature_cols].to_numpy(dtype=float)
y = df[target_col].to_numpy()

# ========= 2) 参数模板 =========
params = {
    "test_size": 0.2,           # 测试集比例
    "random_state": 42,
    "knn_k": 5,                 # KNN邻居数
    "bp_hidden": (64, 32),      # BP分类网络结构
    "bp_max_iter": 500,         # BP迭代次数
    "tree_max_depth": 5         # 决策树最大深度
}

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=params["test_size"], random_state=params["random_state"])

knn = KNeighborsClassifier(n_neighbors=params["knn_k"]).fit(X_train, y_train)
bp = MLPClassifier(hidden_layer_sizes=params["bp_hidden"], max_iter=params["bp_max_iter"], random_state=params["random_state"]).fit(X_train, y_train)
dt = DecisionTreeClassifier(max_depth=params["tree_max_depth"], random_state=params["random_state"]).fit(X_train, y_train)
nb = GaussianNB().fit(X_train, y_train)

print(knn.score(X_test, y_test), bp.score(X_test, y_test), dt.score(X_test, y_test), nb.score(X_test, y_test))


# KNN、BP 神经网络、决策树、朴素贝叶斯

## 输入说明

- 数据文件：默认读取脚本同目录下的 `data.csv`，也可以在代码顶部把 `DATA_FILE` 改为 `.xlsx` 或绝对路径。
- 数据格式：一般要求“一行一个样本/时刻/方案，一列一个变量/指标”。具体列名需要在代码顶部的 `TODO` 参数区填写。
- 示例：若模型需要特征 `特征1、特征2` 和目标列 `y`，表格可整理为：

| 特征1 | 特征2 | y |
|---:|---:|---:|
| 1.2 | 3.4 | 8.1 |
| 2.0 | 2.8 | 9.0 |

## 输出说明

- 控制台会打印核心结果，例如模型参数、评价指标、最优解、排名或预测值。
- 默认结果保存到代码顶部 `OUTPUT_FILE` 指定的文件。
- 若模型包含图形分析，会额外输出图片文件，例如箱型图 `boxplot.png`。

## 原理通俗解释

分类模型 的核心思想是：先把实际问题抽象成可计算的数据结构，再用对应的数学规则寻找“预测值、分类结果、综合得分或最优方案”。代码中已经保留主要计算流程，比赛时重点是把题目数据整理成表格，并把 TODO 参数替换为题目含义一致的列名和约束。

## 适用场景

有标签样本的监督分类，如风险等级、类别识别。

## 局限性

训练集类别不平衡或特征泄漏会导致评估失真。

## 使用提示

- 运行前先检查缺失值、异常值和量纲；很多模型对数据尺度敏感。
- 所有 `TODO` 都应结合题目背景填写，不要直接使用示例列名。
- 建模论文中建议同时写明参数来源，例如权重来自 AHP/熵权法，预测步数来自题目要求。

In [ ]:
"""
KNN、BP 神经网络、决策树、朴素贝叶斯

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "KNN、BP 神经网络、决策树、朴素贝叶斯.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
FEATURE_COLUMNS = ["特征1", "特征2"]  # TODO: 请填写[特征列名列表]，说明：数值型特征。
TARGET_COLUMN = "类别"  # TODO: 请填写[标签列名]，说明：离散分类标签。
TEST_SIZE = 0.2  # TODO: 请填写[测试集比例]，说明：0 到 1。
K_NEIGHBORS = 5  # TODO: 请填写[KNN 邻居数]，说明：正整数。
HIDDEN_LAYERS = (16,)  # TODO: 请填写[BP 隐藏层结构]，说明：样本少时不要过大。
MAX_ITER = 1000  # TODO: 请填写[BP 最大迭代次数]，说明：不收敛时增大。
TREE_MAX_DEPTH = 4  # TODO: 请填写[决策树最大深度]，说明：None 表示不限制。



REQUIRES_DATA = True  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def run_model(data: pd.DataFrame) -> None:
    X = data[FEATURE_COLUMNS]
    y = data[TARGET_COLUMN]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
    models = {
        "KNN": KNeighborsClassifier(n_neighbors=K_NEIGHBORS),
        "BP神经网络": MLPClassifier(hidden_layer_sizes=HIDDEN_LAYERS, max_iter=MAX_ITER, random_state=RANDOM_STATE),
        "决策树": DecisionTreeClassifier(max_depth=TREE_MAX_DEPTH, random_state=RANDOM_STATE),
        "朴素贝叶斯": GaussianNB(),
    }
    rows = []
    for name, clf in models.items():
        pipe = Pipeline([("scaler", StandardScaler()), ("model", clf)])
        pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        rows.append({"模型": name, "准确率": accuracy_score(y_test, pred)})
        print(f"\n{name}\n", classification_report(y_test, pred))
    pd.DataFrame(rows).to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")


if __name__ == "__main__":
    df = load_data()
    run_model(df)
